# AXUM — Ge'ez / Amharic Restoration Training (Kaggle)

Trains the character-level restoration transformer and runs the controlled
Ge'ez vs Amharic comparison on the verse-aligned AGE corpus.

**Baseline to beat: 27.73% top-1** (character n-gram, Ge'ez AGE, 25% damage).

---

## Before you run

In the right-hand sidebar:

1. **Settings -> Accelerator** -> `GPU T4 x2` or `GPU P100`
2. **Settings -> Internet** -> `On`
3. Only if Internet must stay off, **Add Input** the datasets described in Cell 3.

## Two things Kaggle does differently from Colab

**There is no in-cell upload.** `files.upload()` is Colab-only; on Kaggle the
button stays greyed out because the widget never registers. Data reaches a
Kaggle notebook one of two ways only: attached under `/kaggle/input`
(read-only) via **Add Input**, or downloaded over the internet. This notebook
prefers the download route so that nothing needs uploading at all.

**Do not `pip install -r requirements.txt` here.** That file pins
`torch==2.5.1` as a CPU wheel and would replace Kaggle's CUDA build, silently
moving training onto the CPU. Cell 4 installs only the two light packages
Kaggle lacks.

In [ ]:
# 1. Environment check
import subprocess
import sys
from pathlib import Path

import torch

IS_KAGGLE = Path("/kaggle/input").exists()
WORKING = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

print(f"Kaggle           : {IS_KAGGLE}")
print(f"torch            : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM             : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bfloat16 support : {torch.cuda.is_bf16_supported()}")
else:
    print("\nNo GPU. Set Settings -> Accelerator -> GPU before training.")


def has_internet() -> bool:
    """Kaggle blocks outbound traffic unless Internet is switched on."""
    import urllib.request
    try:
        urllib.request.urlopen("https://github.com", timeout=10)
        return True
    except Exception:
        return False


INTERNET = has_internet()
print(f"internet         : {INTERNET}")

In [ ]:
# 2. Locate the AXUM source tree
import shutil

REPO_URL = "https://github.com/girum-work/AXUM.git"
AXUM_ROOT = WORKING / "AXUM"


def find_input(name: str):
    """Search attached Kaggle datasets for a file or folder by name."""
    if not IS_KAGGLE:
        return None
    matches = list(Path("/kaggle/input").rglob(name))
    return matches[0] if matches else None


source_mode = None

if AXUM_ROOT.exists() and (AXUM_ROOT / "scripts").exists():
    source_mode = "already present"
elif INTERNET:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(AXUM_ROOT)],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        source_mode = "git clone"
    else:
        print("Clone failed (private repo?):", result.stderr.strip()[:300])

if source_mode is None:
    # Fall back to an attached copy of the repo.
    attached = find_input("train_restoration.py")
    if attached is not None:
        candidate = attached.parent.parent
        if AXUM_ROOT.exists():
            shutil.rmtree(AXUM_ROOT)
        shutil.copytree(candidate, AXUM_ROOT)
        source_mode = f"copied from {candidate}"

if source_mode is None:
    visible = sorted(p.name for p in Path("/kaggle/input").iterdir()) if IS_KAGGLE else []
    raise FileNotFoundError(
        "No AXUM source found. Either:\n"
        "  A. Switch Settings -> Internet to On so the repo can be cloned, or\n"
        "  B. Add Input a dataset containing the repo (with scripts/ and src/).\n"
        "Note: Kaggle has no in-cell upload; use the Add Input button in the sidebar.\n"
        f"Currently visible under /kaggle/input: {visible or 'none'}"
    )

sys.path.insert(0, str(AXUM_ROOT))
print(f"source: {source_mode}")
print(f"root  : {AXUM_ROOT}")
for required in ("scripts/train_restoration.py", "src/ocr/restoration_model.py",
                 "src/ocr/damage.py", "config.py"):
    print(f"  {'OK ' if (AXUM_ROOT / required).exists() else 'MISSING'}  {required}")

In [ ]:
# 3. Light dependencies only
#
# regex is required, not optional: without it damage.py falls back to a
# hand-rolled grapheme splitter that tokenises Ethiopic differently, so results
# would not match the laptop runs.
for package in ("loguru", "regex"):
    try:
        __import__(package)
        print(f"{package}: present")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        print(f"{package}: installed")

import regex
print("regex version:", regex.__version__)
print("torch still CUDA:", torch.cuda.is_available())

In [ ]:
# 4. Fetch the AGE corpus
#
# AGE is verse-aligned: each record holds the same verse in Ge'ez, Amharic and
# English. Holding content identical is what makes the language comparison
# meaningful; earlier corpora differed in genre or book and confounded it.
%cd {AXUM_ROOT}

CORPUS_SPECS = [
    ("gez", "geez", "data/corpus_raw/geez_age", "data/restoration_corpus_geez_age.json"),
    ("amh", "amharic", "data/corpus_raw/amharic_age", "data/restoration_corpus_amharic_age.json"),
]

for field, language, raw_dir, _ in CORPUS_SPECS:
    target = AXUM_ROOT / raw_dir
    if target.exists() and any(target.glob("*.txt")):
        print(f"{language}: already present")
        continue

    if INTERNET:
        subprocess.run([
            sys.executable, "scripts/fetch_age_dataset.py",
            "--field", field, "--language", language,
            "--output-dir", raw_dir,
        ], check=True)
    else:
        attached = find_input(Path(raw_dir).name)
        if attached is None:
            raise FileNotFoundError(
                f"No corpus for {language}. Switch Internet on, or Add Input a "
                f"dataset containing a folder named '{Path(raw_dir).name}'."
            )
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(attached, target, dirs_exist_ok=True)
        print(f"{language}: copied from {attached}")

In [ ]:
# 5. Chunk the corpora
for _, language, raw_dir, chunked in CORPUS_SPECS:
    if (AXUM_ROOT / chunked).exists():
        print(f"{language}: already chunked")
        continue
    subprocess.run([
        sys.executable, "scripts/prepare_restoration_corpus.py",
        "--input-dir", raw_dir, "--output", chunked,
    ], check=True)

import re

ETHIOPIC = re.compile(r"[\u1200-\u137F\u1380-\u139F\u2D80-\u2DDF\uAB00-\uAB2F]")
for _, language, raw_dir, _ in CORPUS_SPECS:
    text = "".join(f.read_text(encoding="utf-8")
                   for f in sorted((AXUM_ROOT / raw_dir).glob("*.txt")))
    print(f"{language:8s} {len(ETHIOPIC.findall(text)):>9,} Ethiopic characters")

In [ ]:
# 6. Sanity checks before spending GPU time
#
# These guard the two bugs that previously produced wrong numbers: grapheme
# segmentation gluing punctuation onto syllables, and damaged/target sequences
# drifting out of alignment.
from src.ocr.damage import DamageMode, build_training_pair
from src.ocr.restoration_model import FidelVocab

sample = "ወዳዊትሰ ንጉሥ ልህቀ ወኀለፈ መዋዕሊሁ ወይከድንዎ አልባሰ"

mismatch = 0
for mode in DamageMode:
    for seed in range(200):
        damaged, targets = build_training_pair(sample, 0.35, mode=mode, seed=seed)
        mismatch += len(damaged) != len(targets)
print(f"alignment mismatches across all modes : {mismatch}   (must be 0)")

vocab = FidelVocab.from_corpus([sample])
base_ids, vowel_ids = vocab.encode(sample)
print(f"vocabulary round-trip exact           : {vocab.decode(base_ids, vowel_ids) == sample}")

from src.ocr.damage import COMBINING_MARK_RE
print(f"combining-mark range                  : {COMBINING_MARK_RE.pattern}")
print("   expected [\\u135D-\\u135F]; a wider range re-introduces the gluing bug")

assert mismatch == 0, "Damage alignment broken - do not train"
print("\nready to train")

In [ ]:
# 7. Character budget for the matched comparison
#
# Chunk lengths differ between the two corpora (Ge'ez averages ~65 characters,
# Amharic ~33), so equal phrase counts are not equal amounts of text. Budget by
# characters or the comparison measures chunking rather than language.
sys.path.insert(0, str(AXUM_ROOT / "scripts"))
from train_restoration import load_phrases

budgets = {}
for _, language, _, chunked in CORPUS_SPECS:
    phrases = load_phrases(AXUM_ROOT / chunked, 256)
    chars = sum(len(p) for p in phrases)
    budgets[language] = chars
    print(f"{language:8s} {len(phrases):>7,} phrases | {chars:>10,} chars "
          f"| mean {chars / max(len(phrases), 1):5.1f}")

MATCHED_CHARS = min(budgets.values())
print(f"\nmatched budget: {MATCHED_CHARS:,} characters per language")

In [ ]:
# 8. Train Ge'ez
EPOCHS = 60
BATCH_SIZE = 128

!python scripts/train_restoration.py \
    --corpus data/restoration_corpus_geez_age.json \
    --language geez \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --limit-chars {MATCHED_CHARS} \
    --amp --num-workers 2 \
    --out /kaggle/working/models/restoration

In [ ]:
# 9. Train Amharic on the same character budget
!python scripts/train_restoration.py \
    --corpus data/restoration_corpus_amharic_age.json \
    --language amharic \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --limit-chars {MATCHED_CHARS} \
    --amp --num-workers 2 \
    --out /kaggle/working/models/restoration

In [ ]:
# 10. Compare against the n-gram baseline
import json

import matplotlib.pyplot as plt

NGRAM_BASELINE = {"geez": 0.2773, "amharic": 0.2671}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
summary = {}

for language, colour in (("geez", "#c8763c"), ("amharic", "#4c72a8")):
    log_path = AXUM_ROOT / "logs" / "restoration" / f"{language}_training.json"
    if not log_path.exists():
        print(f"no log for {language}")
        continue
    record = json.loads(log_path.read_text(encoding="utf-8"))
    history = record["history"]
    epochs = [h["epoch"] for h in history]

    axes[0].plot(epochs, [h["loss"] for h in history], color=colour, label=language)
    axes[1].plot(epochs, [h["top1"] * 100 for h in history], color=colour, label=language)
    axes[1].axhline(NGRAM_BASELINE[language] * 100, color=colour, linestyle=":",
                    alpha=0.8, label=f"{language} n-gram")

    summary[language] = {
        "best_top1": record["best_top1"],
        "baseline": NGRAM_BASELINE[language],
        "parameters": record["parameters"],
    }

axes[0].set(xlabel="epoch", ylabel="train loss", title="Training loss")
axes[1].set(xlabel="epoch", ylabel="val top-1 (%)", title="Restoration accuracy at 25% damage")
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/kaggle/working/restoration_comparison.png", dpi=150)
plt.show()

print(f"{'language':10s} {'params':>10s} {'n-gram':>9s} {'neural':>9s} {'delta':>9s}")
print("-" * 50)
for language, values in summary.items():
    delta = (values["best_top1"] - values["baseline"]) * 100
    print(f"{language:10s} {values['parameters']:>10,} "
          f"{values['baseline']:>8.2%} {values['best_top1']:>8.2%} {delta:>+8.2f}")

if len(summary) == 2:
    gap = (summary["geez"]["best_top1"] - summary["amharic"]["best_top1"]) * 100
    print(f"\nGe'ez minus Amharic at equal data: {gap:+.2f} points")
    print("n-gram measured this gap at +1.02 points")

In [ ]:
# 11. Collect artefacts for download
#
# Everything under /kaggle/working is attached to the notebook output; anything
# elsewhere disappears when the session ends.
OUTPUT = Path("/kaggle/working")
logs_source = AXUM_ROOT / "logs" / "restoration"
if logs_source.exists():
    shutil.copytree(logs_source, OUTPUT / "logs_restoration", dirs_exist_ok=True)

for path in sorted(OUTPUT.rglob("*")):
    if path.is_file() and path.suffix in {".pth", ".json", ".png"}:
        print(f"  {path.stat().st_size / 1e6:8.2f} MB  {path.relative_to(OUTPUT)}")

print("\nDownload these from the Output tab, or Save Version to persist them.")

## If something goes wrong

**Clone fails** — the repo may be private. Add Input a dataset containing the
repo (it needs `scripts/`, `src/` and `config.py`), or make the repo public.

**`torch.cuda.is_available()` is False** — the accelerator is off, or something
reinstalled torch. Re-check after Cell 4; if a `pip install` replaced it, use
*Factory reset* and avoid installing `requirements.txt`.

**Accuracy sits near 6%** — that was the signature of the target-alignment bug.
Cell 7 asserts against it, so confirm that cell ran and reported 0 mismatches.

**Accuracy plateaus below the n-gram** — likely causes, cheapest first:
`--lr` mistuned for batch 128 (try `1e-3` or `1.5e-4`), too few epochs, or
4.84M parameters being too small (try `--emb-dim 384 --layers 8`).

**Session ends before training finishes** — Kaggle allows 12 hours per session
and 30 GPU hours per week. Reduce `EPOCHS`, or run one language per session;
the two cells are independent.

## Provenance

AGE (`HenokB/AGE-Dataset`) declares no licence upstream. Treat as research use
and cite the authors; do not redistribute the corpus from this notebook.